In [0]:
#################################################
##################stats##########################
#################################################
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, current_timestamp, regexp_extract
)
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

bronze_table = "investment_intelligence_platform.bronze.stats"
silver_table = "investment_intelligence_platform.silver.stats"


df = spark.read.table(bronze_table)

#  extract ticker 
df = df.withColumn(
    "ticker",
    regexp_extract(col("meta_data"), r'([^/]+)\.json$', 1)
)

#  select key columns
silver_df = df.select(
    col("ticker"),
    col("symbol"),
    col("longName"),
    col("sector"),
    col("industry"),
    col("country"),
    col("currency"),
    col("exchange"),
    col("marketCap").cast("double"),
    col("currentPrice").cast("double"),
    col("previousClose").cast("double"),
    col("enterpriseValue").cast("double"),
    col("trailingPE").cast("double"),
    col("forwardPE").cast("double"),
    col("priceToBook").cast("double"),
    col("enterpriseToRevenue").cast("double"),
    col("enterpriseToEbitda").cast("double"),
    col("profitMargins").cast("double"),
    col("grossMargins").cast("double"),
    col("operatingMargins").cast("double"),
    col("returnOnAssets").cast("double"),
    col("returnOnEquity").cast("double"),
    col("revenueGrowth").cast("double"),
    col("earningsGrowth").cast("double"),
    col("totalRevenue").cast("double"),
    col("totalDebt").cast("double"),
    col("totalCash").cast("double"),
    col("freeCashflow").cast("double"),
    col("operatingCashflow").cast("double"),
    col("debtToEquity").cast("double"),
    col("currentRatio").cast("double"),
    col("quickRatio").cast("double"),
    col("dividendRate").cast("double"),
    col("dividendYield").cast("double"),
    col("payoutRatio").cast("double"),
    col("sharesOutstanding").cast("double"),
    col("fullTimeEmployees").cast("double"),
    col("recommendationKey"),
    col("recommendationMean").cast("double"),
    col("numberOfAnalystOpinions").cast("double"),
    col("targetHighPrice").cast("double"),
    col("targetLowPrice").cast("double"),
    col("targetMeanPrice").cast("double"),
    col("targetMedianPrice").cast("double"),
    col("trailingEps").cast("double"),
    col("forwardEps").cast("double"),
    col("bookValue").cast("double"),
    col("last_updated_ts"),
    current_timestamp().alias("silver_updated_ts")
)


#  save with SCD1 
spark.sql("CREATE SCHEMA IF NOT EXISTS investment_intelligence_platform.silver")

def scd_merge_table(spark, source_df, target_table, business_key):
    if not spark.catalog.tableExists(target_table):
        print(f"First load — creating table: {target_table}")
        source_df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)
        print("Table created")
    else:
        print(f"Incremental load — merging: {target_table}")
        merge_condition = " AND ".join(
            [f"target.{c} = source.{c}" for c in business_key]
        )
        delta_table = DeltaTable.forName(spark, target_table)
        delta_table.alias("target") \
            .merge(source_df.alias("source"), merge_condition) \
            .whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
            .execute()
        print("Merge completed ")

scd_merge_table(
    spark,
    silver_df,
    silver_table,
    ["ticker"]
)
print ("silver stats complete")